# UNIDAD 6 — Notebook 3: Implementacion del Proyecto
## Scaffold Adaptable para Cualquier Problema Nano+IA

**Duracion:** 5–8 horas (el nucleo de tu proyecto)  
**Prerequisito:** Plan tecnico validado en U6_02

---

Este notebook es un **scaffold progresivo**. Cada seccion tiene:
1. Una descripcion de lo que debe ir ahi
2. Un ejemplo de referencia (colapsado en comentario) que puedes consultar
3. Un bloque `[ESTUDIANTE]` donde escribes TU implementacion
4. Un checkpoint automatico que valida que completaste la seccion

**Sigue tu propio pipeline definido en U6_02.** No tienes que usar todas las secciones ni en ese orden exacto.

In [1]:
import json
import os
import sys
import warnings
from pathlib import Path
from dotenv import load_dotenv

warnings.filterwarnings("ignore")


def _find_repo_root():
    p = Path.cwd()
    for _ in range(6):
        if (p / ".git").exists() or (p / "environment.yml").exists():
            return p
        p = p.parent
    return Path.cwd()


ROOT = _find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

for _ep in [ROOT / ".env", Path(".env")]:
    if _ep.exists():
        load_dotenv(_ep, override=True)
        print(f"Variables cargadas desde {_ep}")
        break

def get_llm(temperature=0.1):
    # Routing: OpenRouter -> Gemini 2.5 Flash -> OpenAI -> Ollama local
    # Mismo patron que U5_07 / U5_08
    from langchain_openai import ChatOpenAI
    if os.getenv("OPENROUTER_API_KEY"):
        print("LLM: OpenRouter (gemini-2.5-flash)")
        return ChatOpenAI(
            model=os.getenv("OPENROUTER_MODEL", "google/gemini-2.5-flash"),
            base_url="https://openrouter.ai/api/v1",
            api_key=os.getenv("OPENROUTER_API_KEY"),
            temperature=temperature,
            default_headers={
                "HTTP-Referer": "https://github.com/antigravity-nano",
                "X-Title": "Antigravity Nano IA Course",
            },
        )
    try:
        from langchain_google_genai import ChatGoogleGenerativeAI
        if os.getenv("GOOGLE_API_KEY"):
            print("LLM: Google Gemini 2.5 Flash")
            return ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=temperature)
    except ImportError:
        pass
    if os.getenv("OPENAI_API_KEY"):
        print("LLM: OpenAI gpt-4o-mini")
        return ChatOpenAI(model="gpt-4o-mini", temperature=temperature)
    try:
        from langchain_ollama import ChatOllama
        import httpx
        httpx.get("http://localhost:11434/api/tags", timeout=3)
        print("LLM: Ollama local (llama3.2)")
        return ChatOllama(model="llama3.2", temperature=temperature)
    except Exception:
        pass
    print("AVISO: Sin API keys ni Ollama. Seccion 5B requiere al menos una fuente LLM.")
    print("Configura .env con OPENROUTER_API_KEY (ver .env.example en esta carpeta)")
    return None


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def _load_json(path):
    p = Path(path)
    if not p.exists():
        p = ROOT / "educational_content/unit_06_integration_project" / p.name
    if p.exists():
        with open(p, encoding="utf-8") as f:
            return json.load(f)
    return {}


propuesta    = _load_json("mi_proyecto_propuesta.json")
plan_tecnico = _load_json("mi_proyecto_plan_tecnico.json")

TITULO   = propuesta.get("titulo", "[Proyecto sin titulo]").strip()
PREGUNTA = propuesta.get("pregunta_de_investigacion", "").strip()

print(f"Proyecto : {TITULO}")
print(f"Pregunta : {PREGUNTA[:80]}{'...' if len(PREGUNTA) > 80 else ''}")
print(f"Pipeline : {len(plan_tecnico.get('pipeline', []))} etapas definidas")

Variables cargadas desde d:\Users\UCEMICH\Desktop\antigravity projects\IA NANOTECNOLOGIA\Antigravity-Nano-Research-Multiagentic-Core\.env
Proyecto : [ESCRIBE EL TITULO DE TU PROYECTO AQUI]
Pregunta : [ESCRIBE UNA PREGUNTA ESPECIFICA Y RESPONDIBLE]
    Ejemplo: '¿Cual es el tamano...
Pipeline : 3 etapas definidas


---
## Seccion 1: Adquisicion y Exploracion de Datos (EDA) <a id='sec1'></a>

El EDA (Exploratory Data Analysis) es obligatorio antes de modelar. Su objetivo es:
**entender la forma, distribucion, outliers y correlaciones** de tus datos antes de
que un algoritmo los procese.

**Lo que debes producir en esta seccion:**
- Forma del dataset (n_samples, n_features), tipos de datos
- Estadisticas descriptivas (media, std, min, max, percentiles)
- Histogramas y boxplots de las variables clave
- Matriz de correlacion o pairplot

**Conexion con el resto del curso:**
- Tus datos vienen de U1 (estructuras ASE), U2 (simulaciones MD), o datasets externos
- El preprocesamiento de la Seccion 2 depende directamente de lo que encuentres aqui

In [ ]:
# ============================================================
#   [ESTUDIANTE] SECCION 1.1: CARGA DE DATOS
#   Reemplaza el bloque de ejemplo con tu codigo de carga
# ============================================================

# ----- EJEMPLO DE REFERENCIA (comenta o borra cuando uses el tuyo) -----
# df = pd.read_csv("../../notebooks/nanoparticles_dataset.csv")
# -----------------------------------------------------------------------

# [ESTUDIANTE: carga tus datos aqui]
df = None  # Reemplaza con tu carga de datos

# CHECKPOINT 1 — La celda siguiente valida esta seccion
_checkpoint_1_ok = df is not None and isinstance(df, pd.DataFrame) and len(df) > 0

In [ ]:
# ============================================================
#   [ESTUDIANTE] SECCION 1.2: EXPLORACION DE DATOS
#   Describe tus datos: estadisticas, visualizaciones basicas
# ============================================================

if _checkpoint_1_ok:
    print("Shape:", df.shape)
    print("\nPrimeras filas:")
    display(df.head())
    print("\nDescripcion estadistica:")
    display(df.describe())
    print("\nValores nulos:\n", df.isnull().sum())

    # [ESTUDIANTE: agrega visualizaciones relevantes para tu problema]
    # Ejemplo: histograma de la variable objetivo
    # df["tu_variable_objetivo"].hist(bins=30)
    # plt.title("Distribucion de [tu variable]")
    # plt.show()
else:
    print("Checkpoint 1 fallido: el dataframe no esta cargado.")

---
## Seccion 2: Preprocesamiento y Representacion de Caracteristicas <a id='sec2'></a>

Los modelos de ML no entienden datos crudos: entienden numeros en rangos comparables.

**Decisiones clave:**
| Situacion | Tecnica recomendada |
|---|---|
| Features en escalas muy distintas (nm vs eV) | `StandardScaler` o `MinMaxScaler` |
| Variables categoricas (tipo de material) | `OneHotEncoder` o embeddings |
| Estructuras atomisticas como input | Descriptores: Coulomb Matrix, SOAP, ACSF |
| Datos faltantes | Imputacion con mediana o modelos de imputacion |

**Usa `sklearn.pipeline.Pipeline`** para encadenar preprocesamiento + modelo:
esto garantiza que el escalado se aprende SOLO en train, nunca en test.

In [ ]:
# ============================================================
#   [ESTUDIANTE] SECCION 2: PREPROCESAMIENTO
# ============================================================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# [ESTUDIANTE: define tus columnas de caracteristicas y objetivo]
FEATURES = []  # Ej: ["size_nm", "zeta_potential", "composition"]
TARGET   = ""  # Ej: "band_gap_eV"

# Ejemplo de pipeline de preprocesamiento (adapta segun tu caso):
# df_clean = df.dropna(subset=FEATURES + [TARGET])
# X = df_clean[FEATURES].values
# y = df_clean[TARGET].values
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X)
# X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

X_train, X_test, y_train, y_test = None, None, None, None  # Reemplaza con tu codigo

_checkpoint_2_ok = all(x is not None for x in [X_train, X_test, y_train, y_test])
print("Checkpoint 2:", "OK" if _checkpoint_2_ok else "Pendiente — completa el preprocesamiento")

---
## Seccion 3: Entrenamiento del Modelo <a id='sec3'></a>

Implementa el modelo que elegiste en tu plan tecnico (U6_01/U6_02).

**Buenas practicas:**
- Separa train/test **antes** de cualquier preprocesamiento: `train_test_split(X, y, test_size=0.2, random_state=42)`
- Usa `cross_val_score` para estimacion robusta (especialmente con datasets pequeños < 500 muestras)
- Guarda el modelo entrenado con `joblib.dump` o `pickle` — lo necesitaras en U6_04 para el API

**Modelos sugeridos por tipo de problema:**
| Tipo | Primer modelo | Mejor modelo |
|---|---|---|
| Regresion de propiedad | `Ridge` | `GradientBoostingRegressor` |
| Clasificacion de fase | `LogisticRegression` | `RandomForestClassifier` |
| Series temporales | `LinearRegression` | `LightGBM` |
| Imagenes SEM/TEM | `CNN` (U4) | Transfer learning (ResNet) |

In [ ]:
# ============================================================
#   [ESTUDIANTE] SECCION 3: MODELO
#   Implementa el modelo o pipeline que corresponde a tu proyecto
# ============================================================

# --- Ejemplo A: ML clasico (Random Forest) ---
# from sklearn.ensemble import RandomForestRegressor
# model = RandomForestRegressor(n_estimators=200, random_state=42)
# model.fit(X_train, y_train)

# --- Ejemplo B: Red neuronal (PyTorch) ---
# import torch, torch.nn as nn
# class MiRed(nn.Module):
#     def __init__(self, n_in, n_out):
#         super().__init__()
#         self.net = nn.Sequential(nn.Linear(n_in, 128), nn.ReLU(), nn.Linear(128, n_out))
#     def forward(self, x): return self.net(x)
# model = MiRed(X_train.shape[1], 1)

# --- Ejemplo C: Agente LangChain que usa el modelo como herramienta ---
# (ver U5 para el patron completo)

# [ESTUDIANTE: implementa tu modelo aqui]
model = None  # Reemplaza con tu implementacion

_checkpoint_3_ok = model is not None
print("Checkpoint 3:", "OK" if _checkpoint_3_ok else "Pendiente — implementa el modelo")

---
## Seccion 4: Evaluacion y Metricas <a id='sec4'></a>

La evaluacion cierra el loop cientifico: tus metricas deben responder directamente
a la **pregunta de investigacion** que definiste en U6_01.

**Metricas por tipo de problema:**
| Problema | Metrica principal | Metrica secundaria |
|---|---|---|
| Regresion | RMSE (mismas unidades que y) | R2 score |
| Clasificacion balanceada | Accuracy | F1-score |
| Clasificacion desbalanceada | F1 macro | ROC-AUC |
| Ranking / recuperacion | NDCG | MRR |

**Elige tus metricas en funcion del costo del error:**
- Si un falso positivo es costoso (predecir que un material es toxico y no lo es) → maximiza precision
- Si un falso negativo es costoso (perder un candidato prometedor) → maximiza recall

**El resultado de esta seccion es el JSON** `mi_proyecto_resultados.json` que usa U6_04 y U6_05.

In [ ]:
# ============================================================
#   [ESTUDIANTE] SECCION 4: EVALUACION
#   Calcula las metricas de tu criterio de exito
# ============================================================
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score

# [ESTUDIANTE: calcula y guarda tus metricas en el diccionario]
mis_resultados = {
    "metrica_nombre": "",     # Ej: "R2_score"
    "metrica_valor": None,    # Ej: 0.87
    "umbral_exito": None,     # Ej: 0.80 (de tu propuesta)
    "objetivo_superado": None,  # True/False
    "notas": "",              # Observaciones sobre los resultados
}

# Ejemplo de calculo:
# if model is not None and X_test is not None:
#     y_pred = model.predict(X_test)
#     r2 = r2_score(y_test, y_pred)
#     mis_resultados = {
#         "metrica_nombre": "R2_score",
#         "metrica_valor": round(r2, 4),
#         "umbral_exito": 0.80,
#         "objetivo_superado": r2 >= 0.80,
#         "notas": f"El modelo explica {r2*100:.1f}% de la varianza"
#     }

_checkpoint_4_ok = mis_resultados["metrica_valor"] is not None
print("Checkpoint 4:", "OK" if _checkpoint_4_ok else "Pendiente — calcula tus metricas")

if _checkpoint_4_ok:
    print(f"  {mis_resultados['metrica_nombre']}: {mis_resultados['metrica_valor']}")
    print(f"  Objetivo ({mis_resultados['umbral_exito']}): {'SUPERADO' if mis_resultados['objetivo_superado'] else 'NO superado'} ")

In [ ]:
# ============================================================
#   [ESTUDIANTE] SECCION 4.2: VISUALIZACIONES
#   Al menos 2 figuras que ilustren tus resultados
#   Guarda cada figura como .png en la carpeta 'figuras/'
# ============================================================
from pathlib import Path

Path("figuras").mkdir(exist_ok=True)

# [ESTUDIANTE: tus visualizaciones aqui]
# Ejemplo para regresion:
# fig, ax = plt.subplots(figsize=(6, 6))
# ax.scatter(y_test, y_pred, alpha=0.6)
# ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
# ax.set_xlabel("Real")
# ax.set_ylabel("Predicho")
# ax.set_title(f"Prediccion vs Real — R² = {mis_resultados['metrica_valor']}")
# fig.savefig("figuras/prediccion_vs_real.png", dpi=150, bbox_inches='tight')
# plt.show()

print("Genera al menos 2 visualizaciones y guardalas en figuras/")

---
## Seccion 5: Sistema Multi-Agente — Integracion U1-U5 <a id='sec5'></a>

Esta seccion convierte las herramientas de cada unidad en **tools de un agente LLM**
que coordina el pipeline completo de investigacion.

**Arquitectura del sistema:**
```
Agente Coordinador (LangGraph ReAct — U5_02)
    ├── Tool ase_optimizer      ← U1/U2: calcula energia, relaja estructura con ASE/EMT
    ├── Tool ml_predictor       ← U3/U4: predice propiedad con tu modelo entrenado
    ├── Tool literature_search  ← U5_05/U5_06: RAG sobre literatura cientifica
    └── Tool output_scorer      ← U5 external_skills: evalua calidad de resultados
```

**Esta seccion tiene dos partes:**

**5A — Demo ejecutable** (corre ahora, sin API keys, datos sinteticos de Au13):
muestra el patron completo del sistema. Ninguna dependencia externa requerida.

**5B — Tu proyecto** (requiere `.env` con al menos una API key o Ollama local):
conecta tus datos reales, el modelo de tu Seccion 3, y la pregunta de investigacion
de tu propuesta. El agente decide que tools usar y en que orden.

In [2]:
# ============================================================
#   SECCION 5A: DEMO EJECUTABLE
#   Pipeline multi-agente con datos sinteticos de Au13.
#   NO requiere API keys ni modelos entrenados previos.
# ============================================================
import re
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor

# ── TOOL 1: Optimizador de estructura (U1/U2 — ASE/EMT) ─────────────────────
def tool_optimizar_estructura(formula, n_atoms=13):
    # Usa ASE/EMT si disponible, modo simulado como fallback
    try:
        from ase.cluster import Icosahedron
        from ase.calculators.emt import EMT
        from ase.optimize import BFGS
        import io, contextlib
        sym = "".join(c for c in formula if c.isalpha())[:2]
        atoms = Icosahedron(sym, 2)
        atoms.calc = EMT()
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            BFGS(atoms, logfile=None).run(fmax=0.1, steps=20)
        return {"formula": formula, "energia_eV": round(atoms.get_potential_energy(), 4),
                "n_atoms": len(atoms), "metodo": "ASE/EMT"}
    except Exception:
        rng = np.random.default_rng(hash(formula) % 2**31)
        return {"formula": formula, "energia_eV": round(-2.3 * n_atoms + rng.normal(0, 0.1), 4),
                "n_atoms": n_atoms, "metodo": "simulado"}


# ── TOOL 2: Predictor de propiedad (U3/U4 — ML, dataset sintetico) ───────────
_rng = np.random.default_rng(42)
_N = 80
_X_ref = np.column_stack([
    _rng.uniform(1, 20, _N),      # tamano_nm
    _rng.uniform(-50, -10, _N),   # zeta_potential
    _rng.uniform(0.1, 5, _N),     # |energia_eV| por atomo
])
_y_ref = 2.1 + 0.08 * _X_ref[:, 0] - 0.01 * _X_ref[:, 1] + _rng.normal(0, 0.1, _N)
_gbr_demo = GradientBoostingRegressor(n_estimators=50, random_state=42).fit(_X_ref, _y_ref)
print(f"Modelo demo (GBR): R2 = {_gbr_demo.score(_X_ref, _y_ref):.3f}")


def tool_predecir_propiedad(tamano_nm, zeta_potential, energia_eV):
    pred = _gbr_demo.predict([[tamano_nm, zeta_potential, energia_eV]])[0]
    return {"prediccion": round(float(pred), 4), "unidad": "eV (demo)",
            "features": {"tamano_nm": tamano_nm, "zeta_potential": zeta_potential}}


# ── TOOL 3: Consulta de literatura (U5_05/U5_06 — RAG mock) ─────────────────
_PAPERS_DEMO = [
    {"id": "P001", "titulo": "Gold nanoparticle synthesis via citrate reduction", "base": 0.92},
    {"id": "P002", "titulo": "Size-dependent optical properties of Au nanoclusters", "base": 0.87},
    {"id": "P003", "titulo": "EMT potentials for transition metal nanoparticles", "base": 0.81},
    {"id": "P004", "titulo": "Catalytic activity of Au13 on oxide supports", "base": 0.78},
]


def tool_consultar_literatura(query, top_k=2):
    # Mock RAG — en produccion conectar con ChromaDB/Neo4j de U5_05/U5_06
    kw = set(re.findall(r'\w+', query.lower()))
    scored = []
    for p in _PAPERS_DEMO:
        tw = set(re.findall(r'\w+', p["titulo"].lower()))
        scored.append({**p, "score": round(p["base"] * (1 + len(kw & tw) / max(len(kw), 1)), 3)})
    return sorted(scored, key=lambda x: -x["score"])[:top_k]


# ── TOOL 4: Evaluador de calidad (U5 external_skills) ───────────────────────
def tool_evaluar_resultados(resultados):
    try:
        from external_skills.evaluation.output_scorer import OutputScorer
        return {"score": OutputScorer().score(str(resultados)), "metodo": "output_scorer"}
    except Exception:
        score = sum([
            0.25 * ("prediccion" in str(resultados)),
            0.25 * ("energia_eV" in str(resultados)),
            0.25 * ("titulo" in str(resultados)),
            0.25 * ("n_atoms" in str(resultados)),
        ])
        return {"score": round(score, 2), "metodo": "heuristico"}


# ── ORQUESTADOR DEMO (sin LLM — secuencial deterministico) ──────────────────
print("\n" + "=" * 60)
print("DEMO 5A: Pipeline Multi-Agente U1->U2->U3->U5 (sin API keys)")
print("=" * 60)

formula_demo = "Au13"
demo_results = {}

print(f"\n[Tool 1] Optimizando estructura {formula_demo}...")
demo_results["estructura"] = tool_optimizar_estructura(formula_demo, n_atoms=13)
e = demo_results["estructura"]
print(f"  Energia: {e['energia_eV']} eV | n_atoms: {e['n_atoms']} | metodo: {e['metodo']}")

print("\n[Tool 2] Prediciendo propiedad...")
demo_results["prediccion"] = tool_predecir_propiedad(
    1.8, -25.0, abs(e["energia_eV"] / e["n_atoms"])
)
print(f"  Prediccion: {demo_results['prediccion']['prediccion']} {demo_results['prediccion']['unidad']}")

print("\n[Tool 3] Consultando literatura...")
demo_results["papers"] = tool_consultar_literatura(f"{formula_demo} nanoparticle catalysis properties")
for p in demo_results["papers"]:
    print(f"  [{p['score']:.2f}] {p['titulo']}")

print("\n[Tool 4] Evaluando calidad del pipeline...")
demo_results["evaluacion"] = tool_evaluar_resultados(demo_results)
print(f"  Score: {demo_results['evaluacion']['score']} ({demo_results['evaluacion']['metodo']})")

print("\nDemo 5A completada correctamente. Variable demo_results disponible.")

Modelo demo (GBR): R2 = 0.991

DEMO 5A: Pipeline Multi-Agente U1->U2->U3->U5 (sin API keys)

[Tool 1] Optimizando estructura Au13...
  Energia: 6.5997 eV | n_atoms: 13 | metodo: ASE/EMT

[Tool 2] Prediciendo propiedad...
  Prediccion: 2.6245 eV (demo)

[Tool 3] Consultando literatura...
  [1.15] Gold nanoparticle synthesis via citrate reduction
  [1.09] Size-dependent optical properties of Au nanoclusters

[Tool 4] Evaluando calidad del pipeline...
  Score: 1.0 (heuristico)

Demo 5A completada correctamente. Variable demo_results disponible.


---
### Seccion 5B: Agente LLM — Adapta a Tu Proyecto

Ahora conecta las **tools del Demo 5A** con un agente LangGraph real que usa
el LLM para decidir que herramienta llamar y en que orden.

**Requisitos:**
- Al menos una API key en `.env` (ver `.env.example` en esta carpeta)
- O Ollama local gratuito: `ollama pull llama3.2`

**Que adaptar para tu proyecto:**
1. Reemplaza `tool_optimizar_estructura` con tu calculador real de U2
2. Reemplaza `_gbr_demo` con tu modelo entrenado en Seccion 3 (cargado con `joblib.load`)
3. Si tienes ChromaDB de U5_05, conecta `tool_consultar_literatura` al retriever real
4. `PREGUNTA_AGENTE` ya viene de tu propuesta (U6_01) — no necesitas cambiarla

In [5]:

# ============================================================
#   SECCION 5B: AGENTE LangGraph con tools reales
#   Requiere al menos una API key o Ollama corriendo
# ============================================================
import sys
import types

# Guard: si torch no puede inicializar su DLL (WinError 1114 en Windows),
# lo stubbeamos en sys.modules para que langgraph pueda importar sin error.
# Esto NO afecta otros notebooks: sys.modules es local a este kernel.
try:
    import torch  # noqa: F401
    _torch_ok = True
except OSError:
    _torch_ok = False
    for _mod in [
        "torch", "torch.nn", "torch.optim", "torch.utils", "torch.utils.data",
        "transformers", "transformers.modeling_utils",
    ]:
        if _mod not in sys.modules:
            sys.modules[_mod] = types.ModuleType(_mod)
    print("AVISO: torch no pudo inicializar (WinError 1114). Usando stub para langgraph.")
    print("  Seccion 5B funciona con LLM via API. Solo el Ejemplo B (PyTorch) de Sec.3 requiere torch nativo.")

from langchain_core.tools import tool as lc_tool
from langgraph.prebuilt import create_react_agent


# ── Envuelve las tools del Demo 5A como LangChain tools ─────────────────────
@lc_tool
def ase_optimizer(formula: str) -> str:
    "Optimiza la geometria de una estructura nanometrica con ASE/EMT (U1/U2)."
    r = tool_optimizar_estructura(formula)
    return f"Energia {r['formula']}: {r['energia_eV']} eV ({r['metodo']})"


@lc_tool
def ml_predictor(tamano_nm: float, zeta_potential: float, energia_eV_por_atomo: float) -> str:
    "Predice la propiedad objetivo con el modelo ML entrenado en Seccion 3 (U3/U4)."
    r = tool_predecir_propiedad(tamano_nm, zeta_potential, energia_eV_por_atomo)
    return f"Prediccion: {r['prediccion']} {r['unidad']}"


@lc_tool
def literature_search(query: str) -> str:
    "Busca papers relevantes en la base de conocimiento del proyecto (U5_05/U5_06)."
    papers = tool_consultar_literatura(query)
    return "\n".join(f"[{p['score']:.2f}] {p['titulo']}" for p in papers)


# [ESTUDIANTE: agrega tus tools especificas del dominio aqui]
# Ejemplo: conectar un modelo real de U3 via joblib.load
# import joblib
# _mi_modelo = joblib.load("modelo_entrenado.pkl")
# @lc_tool
# def mi_predictor_real(features_csv: str) -> str:
#     "Usa mi modelo real entrenado en Seccion 3."
#     import numpy as np
#     X = np.array([[float(x) for x in features_csv.split(",")]])
#     return f"Prediccion real: {_mi_modelo.predict(X)[0]:.4f}"

tools_proyecto = [ase_optimizer, ml_predictor, literature_search]
# tools_proyecto.append(mi_predictor_real)  # descomenta si tienes modelo real


# ── Crear y ejecutar el agente ───────────────────────────────────────────────
llm = get_llm()

if llm is not None:
    agente = create_react_agent(llm, tools_proyecto)

    PREGUNTA_AGENTE = PREGUNTA if PREGUNTA else (
        "Calcula la energia de un cluster Au13, predice su propiedad principal "
        "y encuentra literatura sobre su actividad catalitica."
    )

    print(f"Pregunta de investigacion:\n{PREGUNTA_AGENTE[:200]}")
    print("-" * 60)

    resultado_agente = agente.invoke({
        "messages": [{"role": "user", "content": PREGUNTA_AGENTE}]
    })

    respuesta = resultado_agente["messages"][-1].content
    print(f"\nRespuesta del agente:\n{respuesta[:800]}")
    componente_avanzado_implementado = True
else:
    componente_avanzado_implementado = True  # Demo 5A ya valida el patron
    print("Demo 5A ejecutada. Para activar 5B: configura .env con OPENROUTER_API_KEY")

print(f"\nSeccion 5: {'Agente LLM activo' if llm else 'Demo 5A completada'}")


LLM: OpenRouter (gemini-2.5-flash)
Pregunta de investigacion:
[ESCRIBE UNA PREGUNTA ESPECIFICA Y RESPONDIBLE]
    Ejemplo: '¿Cual es el tamano optimo de nanoparticulas de ZnO
    para maximizar la degradacion de azul de metileno bajo luz UV?'
------------------------------------------------------------

Respuesta del agente:
¿Cuál es el tamaño óptimo de nanopartículas de ZnO para maximizar la degradación de azul de metileno bajo luz UV?

Seccion 5: Agente LLM activo


In [7]:

# ============================================================
#   RESUMEN DE CHECKPOINTS
# ============================================================
import json

# Valores por defecto si alguna seccion no se ejecuto todavia
_checkpoint_1_ok = _checkpoint_1_ok if '_checkpoint_1_ok' in dir() else False
_checkpoint_2_ok = _checkpoint_2_ok if '_checkpoint_2_ok' in dir() else False
_checkpoint_3_ok = _checkpoint_3_ok if '_checkpoint_3_ok' in dir() else False
_checkpoint_4_ok = _checkpoint_4_ok if '_checkpoint_4_ok' in dir() else False
componente_avanzado_implementado = componente_avanzado_implementado if 'componente_avanzado_implementado' in dir() else False

checkpoints = {
    "1. Datos cargados":         _checkpoint_1_ok,
    "2. Preprocesamiento ok":    _checkpoint_2_ok,
    "3. Modelo implementado":    _checkpoint_3_ok,
    "4. Metricas calculadas":    _checkpoint_4_ok,
    "5. Comp. avanzado":         componente_avanzado_implementado,
}

print("ESTADO DE IMPLEMENTACION")
print("=" * 40)
completados = 0
for nombre, ok in checkpoints.items():
    estado = "[OK]" if ok else "[--]"
    print(f"  {estado}  {nombre}")
    if ok:
        completados += 1

print()
print(f"Progreso: {completados}/{len(checkpoints)} secciones completadas")
minimo_ok = all(checkpoints[k] for k in ["1. Datos cargados", "2. Preprocesamiento ok",
                                          "3. Modelo implementado", "4. Metricas calculadas"])
print()
if minimo_ok:
    print("Implementacion minima completa.")
    print("Guarda los resultados y continua con U6_04_DESPLIEGUE.ipynb")
    # Guardar resultados para notebooks siguientes
    with open("mi_proyecto_resultados.json", "w", encoding="utf-8") as f:
        json.dump(mis_resultados, f, ensure_ascii=False, indent=2)
    print("Resultados guardados en mi_proyecto_resultados.json")
else:
    print("Completa las secciones pendientes antes de continuar.")


ESTADO DE IMPLEMENTACION
  [--]  1. Datos cargados
  [--]  2. Preprocesamiento ok
  [--]  3. Modelo implementado
  [--]  4. Metricas calculadas
  [OK]  5. Comp. avanzado

Progreso: 1/5 secciones completadas

Completa las secciones pendientes antes de continuar.
